# Classical Inference: Body Temperature

Each test is an independent editable block. Change the null value, alternative, alpha, or columns, then rerun that block.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

def find_data_file(file_name):
    candidates = [ROOT / file_name, ROOT / "data" / file_name]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {file_name}. Put it in the project root or data/ folder and include the extension."
    )

def save_plot(fig, file_name):
    path = PLOTS_DIR / file_name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved plot: {path}")
    return path

from classical_toolbox import ClassicalToolbox

use_default_dataset = True
custom_file_name = "my_classical_data.csv"

data_path = ROOT / "data" / "body_temperature.csv" if use_default_dataset else find_data_file(custom_file_name)
tool = ClassicalToolbox(data_path)
df = tool.data.copy()

required_default_columns = ["temperature", "gender"]
if use_default_dataset:
    missing = [c for c in required_default_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Default dataset is missing required columns: {missing}")

print("Dataset:", data_path)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(tool.summarize_data())
print("Columns:", list(df.columns))

In [ ]:
# Editable block: one-sample mean z/t test
column = "temperature"
null_value = 98.6
alternative = "two-sided"
alpha = 0.05
sigma = None  # set a known sigma for a z test, or keep None for a t test

result = tool.one_sample_mean_test(column, null_value, alternative, alpha, sigma, label="One-sample mean test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_one_sample_mean_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_one_sample_mean_ci_from_test.png")

In [ ]:
# Editable block: two-sample mean test
group_col = "gender"
value_col = "temperature"
group_a = sorted(df[group_col].dropna().unique())[0]
group_b = sorted(df[group_col].dropna().unique())[1]
null_difference = 0
alternative = "two-sided"
alpha = 0.05
equal_var = False

result = tool.two_sample_mean_test(group_col, value_col, group_a, group_b, null_difference, alternative, alpha, equal_var, label="Two-sample mean test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_two_sample_mean_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_two_sample_mean_ci.png")

In [ ]:
# Editable block: paired mean test
# Default data are not paired, so the demo creates an after column. For your data, set before_col and after_col to real paired columns.
if use_default_dataset and "temperature_after" not in df.columns:
    paired_shift = np.linspace(-0.05, 0.35, len(df))
    df["temperature_after"] = df["temperature"] + paired_shift
    tool.load_data(df)

before_col = "temperature"
after_col = "temperature_after"
null_difference = 0
alternative = "greater"
alpha = 0.05

result = tool.paired_mean_test(before_col, after_col, null_difference, alternative, alpha, label="Paired mean test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_paired_mean_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_paired_mean_ci.png")

In [ ]:
# Editable block: one-sample variance chi-square test
column = "temperature"
null_variance = 0.5 ** 2
alternative = "two-sided"
alpha = 0.05

result = tool.one_sample_variance_test(column, null_variance, alternative, alpha, label="One-sample variance chi-square test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_one_sample_variance_chi_square_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_one_sample_variance_ci.png")

In [ ]:
# Editable block: two-sample variance F test
group_col = "gender"
value_col = "temperature"
group_a = sorted(df[group_col].dropna().unique())[0]
group_b = sorted(df[group_col].dropna().unique())[1]
null_ratio = 1
alternative = "two-sided"
alpha = 0.05

result = tool.two_sample_variance_test(group_col, value_col, group_a, group_b, null_ratio, alternative, alpha, label="Two-sample variance F test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_two_sample_variance_f_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_two_sample_variance_ci.png")

In [ ]:
# Editable block: one-sample proportion test
# Default demo tests whether the proportion of female observations differs from 0.5.
success_col = "gender"
success_value = "F"
null_probability = 0.5
alternative = "two-sided"
alpha = 0.05

result = tool.one_sample_proportion_test(success_col, success_value, null_probability, alternative, alpha, label="One-sample proportion test")
display(tool.format_result(result))
fig, ax = tool.plot_test_distribution(result)
save_plot(fig, "classical_one_sample_proportion_test.png")
fig, ax = tool.plot_confidence_interval(result)
save_plot(fig, "classical_one_sample_proportion_ci.png")

In [ ]:
# Editable block: mean confidence interval
column = "temperature"
confidence = 0.95
sigma = None

ci = tool.confidence_interval_mean(column, confidence, sigma)
display(pd.DataFrame({"column": [column], "confidence": [confidence], "mean_ci": [ci]}))

In [ ]:
# Editable block: variance confidence interval
column = "temperature"
confidence = 0.95

ci = tool.confidence_interval_variance(column, confidence)
display(pd.DataFrame({"column": [column], "confidence": [confidence], "variance_ci": [ci]}))